<a href="https://colab.research.google.com/github/QaziMahadAhmad/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import glob

print("=== Current Directory Contents ===")
for item in os.listdir('.'):
    print(item)

print("\n=== Searching for notebooks (.ipynb) ===")
for nb in glob.glob('**/*.ipynb', recursive=True):
    print(nb)

print("\n=== Searching for skills folder ===")
for skill in glob.glob('**/skills/**', recursive=True):
    print(skill)

=== Current Directory Contents ===
.config
README (1).md
fact_content_query_90d (1).parquet
dim_content.parquet
sample_data

=== Searching for notebooks (.ipynb) ===

=== Searching for skills folder ===


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:**
The unit of analysis is the unique combination of `(client_hash_id, content_hash_id, query_hash_id)` representing search performance statistics of a specific query for a specific piece of content published under a client.

**Time Window:**
The data spans a rolling historical window representing performance over the past 90 days, tracked and aggregated within the fields (`window_start`, `window_end`). Sub-windows of performance are broken down into `last30` (the last 30 days) and `prev30` (the preceding 30 days, i.e., 30 to 60 days ago).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
import pandas as pd
import numpy as np

fact_path = '/content/fact_content_query_90d (1).parquet'
dim_path = '/content/dim_content.parquet'

df_fact = pd.read_parquet(fact_path)
df_dim = pd.read_parquet(dim_path)

print("=== Fact Table Columns and Types ===")
print(df_fact.dtypes)
print(f"\nFact shape: {df_fact.shape}")

print("\n=== Dimension Table Columns and Types ===")
print(df_dim.dtypes)
print(f"\nDimension shape: {df_dim.shape}")

=== Fact Table Columns and Types ===
client_hash_id                    object
content_hash_id                   object
query_hash_id                     object
query_char_count                   int64
query_token_count                  int64
window_start                      object
window_end                        object
impressions_90d                    int64
clicks_90d                         int64
impressions_last30                 int64
clicks_last30                      int64
impressions_prev30                 int64
clicks_prev30                      int64
avg_position_90d                 float64
avg_position_last30              float64
avg_position_prev30              float64
content_total_impressions_90d      int64
content_visible_query_count        int64
rare_query_count                   int64
rare_impressions_share           float64
anonymized_impressions_share     float64
dtype: object

Fact shape: (2414248, 21)

=== Dimension Table Columns and Types ===
client_hash_id    

In [4]:
print("=== Temporal Range Fields ===")
print(df_fact[['window_start', 'window_end']].drop_duplicates())

grain_cols = ['client_hash_id', 'content_hash_id', 'query_hash_id']
dups = df_fact.duplicated(subset=grain_cols).sum()
print(f"\nDuplicates on {grain_cols}: {dups} out of {len(df_fact)} rows")

print(f"Unique Clients: {df_fact['client_hash_id'].nunique()}")
print(f"Unique Contents: {df_fact['content_hash_id'].nunique()}")
print(f"Unique Queries: {df_fact['query_hash_id'].nunique()}")

=== Temporal Range Fields ===
  window_start  window_end
0   2026-04-02  2026-06-30

Duplicates on ['client_hash_id', 'content_hash_id', 'query_hash_id']: 0 out of 2414248 rows
Unique Clients: 52
Unique Contents: 133852
Unique Queries: 1180090


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classifications
- **Features:**
  - `query_char_count`, `query_token_count`
  - `impressions_90d`, `clicks_90d`
  - `impressions_last30`, `clicks_last30`
  - `impressions_prev30`, `clicks_prev30`
  - `avg_position_90d`, `avg_position_last30`, `avg_position_prev30`
  - `content_total_impressions_90d`, `content_visible_query_count`
  - `rare_query_count`, `rare_impressions_share`
  - `search_volume`, `competition`, `cpc`, `backlinks`
  - `char_count`, `word_count`
- **Labels:**
  - `clicks_last30` or click-through-rate (`clicks_last30 / impressions_last30`) can serve as targets/labels depending on downstream validation criteria.
- **Context:**
  - `client_hash_id`, `content_hash_id`, `query_hash_id`, `keyword_hash_id`, `url_hash_id`
  - `content_type`, `competition_level`, `main_intent`
  - `content_created_date`, `content_updated_date`, `last_optimized_date`
- **Excluded:**
  - `provider_used`, `model_used`: Excluded as these are backend operational details that don't directly determine ranking quality or model targets.
  - `anonymized_impressions_share`: Excluded to prevent skew from unresolvable searches.

In [5]:
# Validate presence of key classification fields and look for null counts
print("=== Missing values in Fact Table ===")
print(df_fact[['impressions_90d', 'clicks_90d', 'avg_position_90d', 'content_total_impressions_90d']].isnull().sum())

print("\n=== Missing values in Dimension Table ===")
cols_to_check = ['search_volume', 'competition', 'cpc', 'backlinks', 'char_count', 'word_count']
print(df_dim[cols_to_check].isnull().sum())

=== Missing values in Fact Table ===
impressions_90d                  0
clicks_90d                       0
avg_position_90d                 0
content_total_impressions_90d    0
dtype: int64

=== Missing values in Dimension Table ===
search_volume    142622
competition      142622
cpc              142622
backlinks        267474
char_count       177768
word_count       177768
dtype: int64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We verify the correctness of the distributions, null values, and window alignments in the code cells.

In [6]:
# Validate window consistency
same_window_for_all = df_fact[['window_start', 'window_end']].nunique() == 1
print(f"Are window_start and window_end uniform across the dataset? \n{same_window_for_all}")

# Confirm that impressions_last30 and impressions_prev30 sum-bounds align reasonably
logical_check = (df_fact['impressions_last30'] + df_fact['impressions_prev30'] <= df_fact['impressions_90d']).all()
print(f"Is (last30 + prev30) impressions <= 90d impressions for all rows? {logical_check}")
if not logical_check:
    violations = (df_fact['impressions_last30'] + df_fact['impressions_prev30'] > df_fact['impressions_90d']).sum()
    print(f"Number of violations: {violations}")

Are window_start and window_end uniform across the dataset? 
window_start    True
window_end      True
dtype: bool
Is (last30 + prev30) impressions <= 90d impressions for all rows? True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limits Identified:**
1. **Temporal Exclusions:** The fact dataset contains performance metrics rolled up over 90 days. It cannot capture fine-grained daily spikes, seasonal day-of-week search trends, or intra-day fluctuations.
2. **Historical Disconnection:** Since dimensions (`dim_content`) are updated and snapshotted statically, historical state changes of content attributes (e.g., historical backlinks or category counts over time) are lost.
3. **Information Overlap:** The `last30` and `prev30` windows represent adjacent but distinct periods. However, comparing them with `90d` results in shared information bounds (mutual overlaps), meaning they are not independent variables.

In [7]:
# Check if we have any mismatch between fact and dim content IDs
fact_content_set = set(df_fact['content_hash_id'].unique())
dim_content_set = set(df_dim['content_hash_id'].unique())

missing_in_dim = len(fact_content_set - dim_content_set)
missing_in_fact = len(dim_content_set - fact_content_set)

print(f"Content IDs in Fact but missing in Dim: {missing_in_dim}")
print(f"Content IDs in Dim but missing in Fact: {missing_in_fact}")

Content IDs in Fact but missing in Dim: 0
Content IDs in Dim but missing in Fact: 385754


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.